Changing The Directoryto Root Directory

In [1]:
%pwd

'C:\\Users\\DELL\\Desktop\\New folder (8)\\New folder (2)\\medibot\\ai-medical-bot\\research'

In [2]:
import os 
os.chdir("../")

In [3]:
%pwd

'C:\\Users\\DELL\\Desktop\\New folder (8)\\New folder (2)\\medibot\\ai-medical-bot'

In [5]:
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
def document_loader(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader
                           )
    text=loader.load()
    return text

In [7]:
extracted_data=document_loader("Data/")

In [8]:
#extracted_data

In [9]:
def text_split(text):
    ts=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text=ts.split_documents(text)
    return text

In [10]:
chunk= text_split(extracted_data)
print("Length of Text Chunks", len(chunk))

Length of Text Chunks 5860


Loading Hugging Face

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

In [12]:
def download_HuggingFace():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [15]:
embeddings = download_HuggingFace()

In [16]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [31]:
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
PINECONE_API_KEY=os.environ.get("PINECONE_API_KEY")
gemini_api_key= os.environ.get("gemini_api_key")

In [33]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

In [35]:
index_name = "medibot"


pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws", 
        region="us-east-1"
    ) 
    )

{
    "name": "medibot",
    "metric": "cosine",
    "host": "medibot-bqfc1fc.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
  

In [36]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["gemini_api_key"] = gemini_api_key

In [45]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
#old
from langchain_pinecone import Pinecone
'''
nw from langchain_pinecone import PineconeVectorStore
pip install -U langchain-pinecone'''
docsearch = Pinecone.from_documents(
    documents=chunk,
    index_name=index_name,
    embedding=embeddings, 
)

In [46]:
import langchain_pinecone
print(dir(langchain_pinecone))

['Pinecone', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_utilities', 'vectorstores']


In [51]:
from langchain_pinecone import Pinecone

docsearch= Pinecone.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [52]:
docsearch

In [53]:
retriever=docsearch.as_retriever(searchtype="similarity",  search_kwargs={'k':3})

In [54]:
retrived1=retriever.invoke('What is acne')
retrived1

[Document(metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 37, 'page_label': '38', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin dis

In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(temperature=0.7, max_tokens=500,model="gemini-3-flash-preview", )

In [68]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt=(
"You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}")

prompt=ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human","{input}")])

In [70]:
qna=create_stuff_documents_chain(llm,prompt)
rag_chain=create_retrieval_chain(retriever,qna)

In [71]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly and gigantism are disorders caused by the abnormal release of growth hormone from the pituitary gland, leading to increased growth in bone and soft tissue. Gigantism occurs when this abnormality happens before bone growth stops, resulting in unusual height. Acromegaly occurs when the hormone excess happens after bone growth has finished, typically affecting middle-aged adults.
